In [64]:
# Imports

from __future__ import annotations

import heapq
import math
import random
import sys
import time
from typing import List, Set, Tuple

In [65]:
# GraphData class

class GraphData:
    P2Graph2AdjList: List[List[int]] = [
        [1, 4, 8],  # 0 <- This vertex is shared with the previous graph in the chain
        [0, 2, 9],  # 1
        [1, 3, 5],  # 2 <- This vertex is shared with the next graph in the chain
        [2, 4, 6],  # 3
        [0, 3, 7],  # 4
        [2, 7, 8],  # 5
        [3, 8, 9],  # 6
        [4, 5, 9],  # 7
        [0, 5, 6],  # 8
        [1, 6, 7],  # 9
    ]


In [66]:
# GraphBuilder class

class GraphBuilder:
    
    @staticmethod
    def ExtendGraph(
        graphAdjList: List[List[int]],
        oldConnectionPoint: int,
        newConnectionPoint: int,
        chainLength: int,
    ) -> List[List[int]]:
        adjList: List[List[int]] = [list(t) for t in graphAdjList]

        for i in range(chainLength - 1):
            oldSharedPoint = oldConnectionPoint + 9 * i
            newSharedPoint = newConnectionPoint + 9 * i

            for baseVertexIndex in range(len(graphAdjList)):
                newVertexIndex = baseVertexIndex + 9 * (i + 1)

                for neighborIndex in graphAdjList[baseVertexIndex]:
                    if baseVertexIndex == newConnectionPoint:
                        newNeighborIndex = neighborIndex + 9 * (i + 1)
                        adjList[oldSharedPoint].append(newNeighborIndex)
                    else:
                        # Instantiate if it does not exist.
                        if len(adjList) <= newVertexIndex:
                            adjList.append([])

                        if neighborIndex == newConnectionPoint:
                            adjList[newVertexIndex].append(oldSharedPoint)
                        else:
                            newNeighborIndex = neighborIndex + 9 * (i + 1)
                            adjList[newVertexIndex].append(newNeighborIndex)

        return [list(inner) for inner in adjList]

    @staticmethod
    def BuildChain(
        originalAdj: List[List[int]],
        idxOld: int,
        idxNew: int,
        repeat: int,
    ) -> List[List[int]]:
        if originalAdj is None:
            raise ValueError("originalAdj cannot be None")

        if idxOld < 0 or idxOld >= len(originalAdj):
            raise ValueError("idxOld out of range")

        if idxNew < 0 or idxNew >= len(originalAdj):
            raise ValueError("idxNew out of range")

        if repeat < 0:
            raise ValueError("repeat cannot be negative")

        n = len(originalAdj)  # vertices in one copy
        totalClones = repeat + 1  # how many copies we will end up with
        totalVertices = n * totalClones - repeat  # we merge one vertex for every link

        # Work with a mutable list of vertices
        vertices: List[List[int]] = []

        # Helper: clone the original adjacency list (deep copy)
        def CloneGraph(graph: List[List[int]]) -> List[List[int]]:
            return [list(row) for row in graph]

        # 1. Add the first clone (the original graph)
        vertices.extend(CloneGraph(originalAdj))

        # 2. For each subsequent clone, merge the two chosen vertices
        offset = n  # first free index in the growing graph

        for step in range(repeat):
            # Clone the original graph again
            clone = CloneGraph(originalAdj)

            # The indices of the vertices that will be merged
            mergeOld = idxOld  # in the current growing graph
            mergeNew = idxNew + offset  # in the fresh clone (after shifting)

            # ---- Build merged adjacency for the shared vertex ----
            mergedAdj: Set[int] = set()

            # old adjacency
            for v in vertices[mergeOld]:
                if v != mergeNew:
                    mergedAdj.add(v)

            # new adjacency (shifted)
            for v in clone[idxNew]:
                shifted = v + offset
                if shifted != mergeOld:
                    mergedAdj.add(shifted)

            vertices[mergeOld] = sorted(mergedAdj)

            # ---- Replace references to the new vertex with the old one ----
            for i in range(len(vertices)):
                for j in range(len(vertices[i])):
                    if vertices[i][j] == mergeNew:
                        vertices[i][j] = mergeOld

            # ---- Add the remaining vertices of the new clone ----
            for i in range(n):
                if i == idxNew:
                    continue  # already merged

                # shift adjacency indices
                shiftedAdj = [v + offset for v in clone[i]]
                vertices.append(shiftedAdj)

            # Update the offset for the next clone
            offset += n

        # Convert back to an array for the caller
        return vertices

In [67]:
# Python conversion of the C# GraphLabeler.cs code

class GraphLabeler:

    def __init__(self, adjList: List[List[int]]):
        self.adjList = adjList
        self.labels: List[int] = [0] * len(adjList) # 0 means unlabeled

        # Note on conversion: A C# HashSet's add() function will
        # "return true if the element is added to the HashSet<T> object; false if the element is already present."
        # But in Python you have to first check if not value in set then set.add(value) 
        self.edgeSet: Set[int] = set()

    def SolveLabels(self) -> List[int]:
        # TODO: Potential optimization:
        #       Prioritize first by lowest label *then* by largest number of labeled adjacent nodes. 
        #       In C# use OrderedDictionary of labeled frontier nodes keyed by label, then SortedSet?
        frontier: List[tuple[int, int]] = []  # priority queue of (priority, vertex_id)
        heapq.heapify(frontier)

        self.labels[0] = 1  # Set first vertex label to be 1
        for i in range(len(self.adjList[0])):
            heapq.heappush(frontier, (1, self.adjList[0][i]))  # Add starting vertexes

        skippedWeights: List[int] = []
        nextLargestWeight = 2

        while frontier:
            # Get vertex to be labeled
            currentVertexId = heapq.heappop(frontier)[1]
            #_ = frontier[0][0] if frontier else None  # commented out block

            # Don't label already labeled nodes
            if self.labels[currentVertexId] != 0:
                continue

            # Get adjacent vertexes
            adjVertexes = self.adjList[currentVertexId]

            # Labels will be stored in accenting order
            adjLabels: List[int] = []

            unlabeledNeighbors: List[int] = []

            # Get the labels of adjacent vertexes if set AKA non-zero
            for i in range(len(adjVertexes)):
                label = self.labels[adjVertexes[i]]
                if label > 0:
                    adjLabels.append(label)
                else:
                    unlabeledNeighbors.append(adjVertexes[i])

            adjLabels.sort()  # Sort smallest to largest for efficiency

            # Assert there is at least one labeled adjacent vertex
            if not adjLabels:
                raise Exception("Labeling vertex without labeled neighbors!")


            # Find a label for the current vertex 

            newLabel = None

            # Try skipped weights to make a valid label
            weightIndex = 0
            while weightIndex < len(skippedWeights):
                nextEdge = skippedWeights[weightIndex]
                potentialLabel = nextEdge - adjLabels[0]

                if self.CheckLabel(potentialLabel, currentVertexId, adjLabels):
                    # Label is valid
                    newLabel = potentialLabel
                    skippedWeights.pop(weightIndex)
                    break
                weightIndex += 1

            # Use max weight to make a valid label
            while newLabel is None:
                # Note: labelsOfAdjNodes[0] *should* be the smallest for efficiency
                potentialLabel = nextLargestWeight - adjLabels[0]

                if self.CheckLabel(potentialLabel, currentVertexId, adjLabels):
                    newLabel = potentialLabel
                else:
                    # Put skipped values in skippedWeights in ascending order.
                    # Since new values are always larger than previous values
                    # append() works and no further logic is needed.
                    skippedWeights.append(nextLargestWeight)

                nextLargestWeight += 1

            # Assert new label must picked
            if newLabel is None or newLabel == 0:
                raise Exception("No valid label found!")

            # Set new label
            self.labels[currentVertexId] = newLabel

            # Add adjNodes that have not been labeled
            for unlabeledNeighbor in unlabeledNeighbors:
                heapq.heappush(frontier, (newLabel, unlabeledNeighbor))

        # Assert All nodes should be labeled when the queue is empty.
        if 0 in self.labels:
            # throw Exception("Some vertexes were not labeled!");
            print("Some vertexes were not labeled!", file=sys.stderr)

        return self.labels


    def CheckLabel(self, label: int, currentVertexIndex: int, adjLabels: List[int]) -> bool:
        newEdges: List[int] = []

        for adjLabel in adjLabels:
            newEdge = label + adjLabel
            if newEdge in self.edgeSet or newEdge in newEdges:
                return False
            newEdges.append(newEdge)

        # Block if this label is used two degrees away
        if self.CheckTwoDegreeDuplicate(label, currentVertexIndex):
            return False

        for newEdge in newEdges:
            if newEdge in self.edgeSet:
                raise Exception(f"Edge {newEdge} already exists!")
            self.edgeSet.add(newEdge)

        return True

    # returns True if a duplicate exists; False if no duplicate was found.
    def CheckTwoDegreeDuplicate(self, potentialLabel: int, vertexIndex: int) -> bool:
        for firstNeighborIndex in self.adjList[vertexIndex]:
            for secondNeighborIndex in self.adjList[firstNeighborIndex]:
                if potentialLabel == self.labels[secondNeighborIndex]:
                    return True

        return False



In [68]:
# MermaidGraph helper class
# Generates mermaid graphs from an adjacency list and optionally labels.

class MermaidGraph:
    """
    Builds a Mermaid diagram (flowchart LR) from an adjacency-list.
    Nodes are labelled with the integer array ``vertex_labels``.
    Each undirected edge is annotated with the sum of the two vertex
    indices it connects.
    """

    @staticmethod
    def generate_mermaid_graph(
        adjacency_list: List[List[int]], vertex_labels: List[int]
    ) -> str:
        """
        :param adjacency_list: adjacencyList[i] contains a list of vertex indices adjacent to vertex i.
                               The list may contain each edge twice (because the graph is undirected).
        :param vertex_labels: Integer labels for each vertex. vertex_labels[i] will be shown for node i.
        :return: A string that can be pasted into a Mermaid live editor or Markdown.
        """
        # --- Validation (mimics ArgumentNullException / ArgumentException) ---
        if adjacency_list is None:
            raise ValueError("adjacency_list cannot be None")
        if vertex_labels is None:
            raise ValueError("vertex_labels cannot be None")
        if len(adjacency_list) != len(vertex_labels):
            raise ValueError(
                "Adjacency list and label array must contain the same number of vertices."
            )

        # --- Build the diagram string -------------------------------------
        sb: List[str] = []

        # 1. Header – top‑down layout (feel free to change to graph LR, etc.)
        sb.append("---")
        sb.append("config:")
        sb.append("  layout: fixed")
        sb.append("---")
        sb.append("flowchart LR")

        # 2. Define the nodes with their integer labels.
        for i in range(len(adjacency_list)):
            # Node id: V0, V1, …   (any unique string works)
            sb.append(f"    V{i}[[{vertex_labels[i]}]]")  # double brackets for a “rounded rectangle” style

        # 3. Add the undirected edges – only once per edge.
        seen_edges: Set[Tuple[int, int]] = set()
        seen_edge_weights: Set[int] = set()
        for u in range(len(adjacency_list)):
            if adjacency_list[u] is None:
                continue
            for v in adjacency_list[u]:
                # sanity checks
                if v < 0 or v >= len(adjacency_list) or u == v:
                    continue
                # canonical order ensures we never output the same edge twice
                key = (u, v) if u < v else (v, u)
                if key in seen_edges:
                    continue
                seen_edges.add(key)
                edge_sum = vertex_labels[u] + vertex_labels[v]
                if edge_sum in seen_edge_weights:
                    raise ValueError("Duplicate edge weight encountered.")
                seen_edge_weights.add(edge_sum)
                sb.append(f"    V{u}<-->|{edge_sum}|V{v}")

        return "\n".join(sb) + "\n"

    @staticmethod
    def generate_mermaid_graph_no_labels(
        adjacency_list: List[List[int]]
    ) -> str:
        """
        Variant that uses the vertex index as the label for each node.
        Equivalent to ``GenerateMermaidGraph(int[][] adjacencyList)`` in C#.
        """
        # --- Validation -------------------------------------------------------
        if adjacency_list is None:
            raise ValueError("adjacency_list cannot be None")

        # --- Build the diagram string ---------------------------------------
        sb: List[str] = []

        # 1. Header – top‑down layout (feel free to change to graph LR, etc.)
        sb.append("---")
        sb.append("config:")
        sb.append("  layout: fixed")
        sb.append("---")
        sb.append("flowchart LR")

        # 2. Define the nodes with their integer labels.
        for i in range(len(adjacency_list)):
            sb.append(f"    V{i}[[{i}]]")  # double brackets for a “rounded rectangle” style

        # 3. Add the undirected edges – only once per edge.
        seen_edges: Set[Tuple[int, int]] = set()
        for u in range(len(adjacency_list)):
            if adjacency_list[u] is None:
                continue
            for v in adjacency_list[u]:
                # sanity checks
                if v < 0 or v >= len(adjacency_list) or u == v:
                    continue
                # canonical order ensures we never output the same edge twice
                key = (u, v) if u < v else (v, u)
                if key in seen_edges:
                    continue
                seen_edges.add(key)
                sb.append(f"    V{u}<-->V{v}")

        return "\n".join(sb) + "\n"


# --------------------------------------------------------------------------
# Example usage (not part of the library but shows how the class is used)
# --------------------------------------------------------------------------
"""
if __name__ == "__main__":
    # Sample graph (undirected, edges may appear twice)
    adjacency = [
        [1, 2],   # 0
        [0, 2],   # 1
        [0, 1],   # 2
    ]
    labels = [10, 20, 30]
    print(MermaidGraph.generate_mermaid_graph(adjacency, labels))

    # Without explicit labels (vertices are labelled by their indices)
    print(MermaidGraph.generate_mermaid_graph_no_labels(adjacency))
"""
suppressCellOutput = True

In [ ]:
# Helper methods to run the labeling algorithm

def RunOnce(chain_length: int, printGraph: bool = False) -> List[int]:
    """
    Executes a single labeling run on a graph of the specified chain length.

    Parameters
    ----------
    chain_length : int
        Length of the chain to be constructed.

    Returns
    -------
    List[int]
        The resulting vertex labels.
    """
    startTime = time.time()

    # Build the graph – identical to C#:
    # var graph = GraphBuilder.ExtendGraph(GraphData.P2Graph2AdjList, 2, 0, chainLength);
    graph = GraphBuilder.ExtendGraph(
        GraphData.P2Graph2AdjList, 2, 0, chain_length
    )

    graphBuildTime = time.time() - startTime
    startLabelerTime = time.time() # Reset "timer"

    # Create the labeler and solve
    labeler = GraphLabeler(graph)
    labels = labeler.SolveLabels()

    labelerTimeElapsed = time.time() - startLabelerTime
    totalTimeElapsed = time.time() - startTime

    # Output the min‑k and max‑label (mirrors Console.WriteLine in C#)
    min_k = math.ceil(((chain_length * 15 + 1) / 2))
    print(f"Min-k: {min_k}, Max-Label: {max(labels)}")

    # Optionally print of the graph as a Mermaid graph
    if printGraph:
        # Console.Write("Labels: ");
        print("Labels: ", end="")
        for label in labels:
            print(f"{label},", end="")
        print()

        # Console.Write("Edges: ");
        print("Edges: ", end="")
        for edgeWeight in labeler.edgeSet:
            print(f"{edgeWeight},", end="")
        print()

        mermaid = MermaidGraph.generate_mermaid_graph(graph, labels)
        # mermaid = MermaidGraph.generate_mermaid_graph(graph)

        print()
        print()
        print(mermaid)

        print()
        print(F"{graphBuildTime:.6f} seconds - Time to build graph")
        print(F"{labelerTimeElapsed:.6f} seconds - Time to label graph")
        print(F"{totalTimeElapsed:.6f} seconds - Total time")

    # Check for duplicate edges
    edgeWeightSet: set[int] = set()
    for edgeWeight in labeler.edgeSet:
        if edgeWeight in edgeWeightSet:
            print(f"Duplicate edge found #{edgeWeight}")
        else:
            edgeWeightSet.add(edgeWeight)

    return labels


def RunRange(min_chain: int, max_chain: int, printGraph: bool = False) -> None:
    """
    Runs the labeling algorithm for every chain length in the inclusive
    range [min_chain, max_chain] and prints the results.
    """
    for i in range(min_chain, max_chain + 1):
        RunOnce(i, printGraph)



In [70]:
# Debug functions, please ignore.

def RunFindOptimal() -> None:
    """
    Tries to find an optimal labeling by repeatedly shuffling the
    provided label array and attempting to generate a valid Mermaid
    diagram.  The first successful diagram is printed.
    """
    rnd = random.Random()
    labels = [1, 1, 2, 4, 5, 7, 2, 8, 8, 7]

    for _ in range(999_999):
        try:
            mermaid = MermaidGraph.generate_mermaid_graph(
                GraphData.P2Graph2AdjList, labels
            )
            print(mermaid)
            break
        except Exception:
            # ignored
            pass

        rnd.shuffle(labels)

    # Make labeling order fixed, based on this graph?
    # Function result/output:
    """
    ---
    config:
      layout: fixed
    ---
    flowchart LR
        V0[[1]]
        V1[[4]]
        V2[[7]]
        V3[[5]]
        V4[[8]]
        V5[[7]]
        V6[[2]]
        V7[[8]]
        V8[[1]]
        V9[[2]]
        V0<-->|5|V1
        V0<-->|9|V4
        V0<-->|2|V8
        V1<-->|11|V2
        V1<-->|6|V9
        V2<-->|12|V3
        V2<-->|14|V5
        V3<-->|13|V4
        V3<-->|7|V6
        V4<-->|16|V7
        V5<-->|15|V7
        V5<-->|8|V8
        V6<-->|3|V8
        V6<-->|4|V9
        V7<-->|10|V9
    """

In [73]:
RunOnce(1, True)

# RunRange(1, 3)

Min-k: 8, Max-Label: 13
Labels: 1,1,2,6,3,8,13,8,4,5,
Edges: 2,3,4,5,6,8,9,10,11,12,13,16,17,18,19,



---
config:
  layout: fixed
---
flowchart LR
    V0[[1]]
    V1[[1]]
    V2[[2]]
    V3[[6]]
    V4[[3]]
    V5[[8]]
    V6[[13]]
    V7[[8]]
    V8[[4]]
    V9[[5]]
    V0<-->|2|V1
    V0<-->|4|V4
    V0<-->|5|V8
    V1<-->|3|V2
    V1<-->|6|V9
    V2<-->|8|V3
    V2<-->|10|V5
    V3<-->|9|V4
    V3<-->|19|V6
    V4<-->|11|V7
    V5<-->|16|V7
    V5<-->|12|V8
    V6<-->|17|V8
    V6<-->|18|V9
    V7<-->|13|V9


0.000009 - Time to build graph
0.000032 - Time to label graph
0.000042 - Total time


[1, 1, 2, 6, 3, 8, 13, 8, 4, 5]